# 01 · Baselines: full fine-tuning and head-only (English → Swahili zero-shot NER)

Trains XLM-R on **English CoNLL-2003** and evaluates **zero-shot on MasakhaNER 2.0 Swahili**.
Model selection uses the **English dev set only**; the Swahili test set is scored once, at the end.

**Run it top to bottom, one cell at a time.** On Colab: *Runtime → Change runtime type → GPU (T4 is fine)*.
Change `METHOD` / `SEED` in section 2 to run the other baseline or another seed.

| Section | What happens |
|---|---|
| 1 | Environment (Colab: mount Drive, clone repo, install) |
| 2 | Config |
| 3 | Data download, label mapping, tokenization check |
| 4 | Model + trainable-parameter count |
| 5 | Training (selection on English dev F1) |
| 6 | Evaluation: English dev/test, Swahili test |
| 7 | Save cost + metrics, peek at Swahili errors |

## 1. Environment

In [1]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Ande404/FrozenWeights.git"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO = Path("/content/FrozenWeights")
    if not REPO.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO), "pull"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements.txt")], check=True)
    # Checkpoints + logs go to Drive so a dropped Colab session doesn't lose them.
    OUTPUT_ROOT = Path("/content/drive/MyDrive/FrozenWeights/outputs")
else:
    REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    OUTPUT_ROOT = REPO / "outputs"

os.chdir(REPO)
sys.path.insert(0, str(REPO / "src"))
print("repo:", REPO, "| outputs:", OUTPUT_ROOT)

repo: /Users/luwaga/Desktop/Projects/FrozenWeights/FrozenWeights | outputs: /Users/luwaga/Desktop/Projects/FrozenWeights/FrozenWeights/outputs


In [2]:
import json, platform
import numpy as np, pandas as pd, torch, transformers, yaml
from transformers import (AutoTokenizer, DataCollatorForTokenClassification, Trainer,
                          TrainingArguments, set_seed)

from frozenweights import cost, data, metrics, models

print(f"python {platform.python_version()} | torch {torch.__version__} | transformers {transformers.__version__}")
print("device:", cost.get_device(), "|", cost.device_name())

python 3.11.0 | torch 2.14.1 | transformers 4.57.6
device: mps | Apple MPS (arm)


## 2. Config

Defaults come from `configs/<METHOD>.yaml`. `MAX_TRAIN` lets you do a quick smoke run (e.g. `500`); leave it `None` for real runs.

In [3]:
METHOD = "full_ft"      # "full_ft" | "head_only" | "lora"
SEED = 1
MAX_TRAIN = None         # e.g. 500 for a smoke test

cfg = yaml.safe_load(open(REPO / "configs" / f"{METHOD}.yaml"))
RUN_NAME = f"{METHOD}_seed{SEED}" + (f"_n{MAX_TRAIN}" if MAX_TRAIN else "")
RUN_DIR = OUTPUT_ROOT / RUN_NAME
# On Colab, results go to Drive too: /content is wiped when the session ends.
RESULTS_DIR = (OUTPUT_ROOT.parent if IN_COLAB else REPO) / "results" / "checkpoint1"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
set_seed(SEED)
cfg

{'method': 'full_ft',
 'model_name': 'xlm-roberta-base',
 'max_length': 256,
 'learning_rate': 2e-05,
 'num_train_epochs': 3,
 'per_device_train_batch_size': 32,
 'per_device_eval_batch_size': 64,
 'warmup_ratio': 0.1,
 'weight_decay': 0.01,
 'seeds': [1, 2, 3]}

## 3. Data

`scripts/download_data.py` fetches the raw CoNLL files and writes `data/processed/{en,sw}/{train,dev,test}.jsonl`.
Labels are mapped to the shared space **PER / ORG / LOC** (CoNLL `MISC` → `O`, MasakhaNER `DATE` → `O`).

In [4]:
if not (data.PROCESSED_DIR / "sw" / "test.jsonl").exists():
    subprocess.run([sys.executable, "scripts/download_data.py"], check=True)

en = {s: data.load_split("en", s) for s in data.SPLITS}
sw = {s: data.load_split("sw", s) for s in data.SPLITS}
if MAX_TRAIN:
    en["train"] = en["train"][:MAX_TRAIN]

print("labels:", data.LABELS)
pd.DataFrame({f"{lang}/{s}": [len(rows)] for lang, d in (("en", en), ("sw", sw)) for s, rows in d.items()}, index=["sentences"])

labels: ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']


,en/train,en/dev,en/test,sw/train,sw/dev,sw/test
sentences,14041,3250,3453,6593,942,1883


In [5]:
# One example per language: original tags vs. mapped tags
for lang, rows in (("en", en["train"]), ("sw", sw["test"])):
    ex = next(r for r in rows if any(t[2:] in ("MISC", "DATE") for t in r["orig_tags"]))
    print(f"[{lang}]")
    print(pd.DataFrame({"token": ex["tokens"], "orig": ex["orig_tags"], "mapped": ex["tags"]}).T.to_string(header=False), "\n")

[en]
token      EU  rejects  German  call  to  boycott  British  lamb  .
orig    B-ORG        O  B-MISC     O   O        O   B-MISC     O  O
mapped  B-ORG        O       O     O   O        O        O     O  O 

[sw]
token   Shujaa  Mwakilishi  wa  Baraza  la  Wawakilishi   John  Lewis  atazikwa  Alhamisi  baada  ya  ibada  ya  maziko  yake  huko  Atlanta  .
orig         O           O   O       O   O            O  B-PER  I-PER         O    B-DATE      O   O      O   O       O     O     O    B-LOC  O
mapped       O           O   O       O   O            O  B-PER  I-PER         O         O      O   O      O   O       O     O     O    B-LOC  O 



In [6]:
tokenizer = AutoTokenizer.from_pretrained(cfg["model_name"])
ML = cfg["max_length"]
train_ds = data.to_hf_dataset(en["train"], tokenizer, ML)
eval_sets = {
    "en_dev": data.to_hf_dataset(en["dev"], tokenizer, ML),
    "en_test": data.to_hf_dataset(en["test"], tokenizer, ML),
    "sw_test": data.to_hf_dataset(sw["test"], tokenizer, ML),
}

# Show how a Swahili sentence is split into sub-words and where the label sits (first sub-word only)
enc = data.tokenize_and_align(sw["test"][:1], tokenizer, ML)
pieces = tokenizer.convert_ids_to_tokens(enc["input_ids"][0])
print(pd.DataFrame({"subword": pieces, "label": [data.ID2LABEL.get(l, "·") for l in enc["labels"][0]]}).T.to_string(header=False))
print({k: len(v) for k, v in {"train": train_ds, **eval_sets}.items()})

subword  <s>  ▁Shu  ja  a  ▁Mwakilishi  ▁wa  ▁Baraza  ▁la  ▁Wa  wak  ilishi  ▁John  ▁Lewis  ▁ata  zik  wa  ▁Alhamisi  ▁baada  ▁ya  ▁i  bada  ▁ya  ▁ma  ziko  ▁yake  ▁huko  ▁Atlanta  ▁  .  </s>
label      ·     O   ·  ·            O    O        O    O    O    ·       ·  B-PER   I-PER     O    ·   ·          O       O    O   O     ·    O    O     ·      O      O     B-LOC  O  ·     ·
{'train': 14041, 'en_dev': 3250, 'en_test': 3453, 'sw_test': 1883}


## 4. Model

In [7]:
model = models.build_model(METHOD, cfg["model_name"], cfg)
params = cost.count_parameters(model)
print(f"total {params['total_params']:,} | trainable {params['trainable_params']:,} ({params['trainable_pct']:.4f}%)")

Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


total 277,458,439 | trainable 277,458,439 (100.0000%)


## 5. Train

Evaluates on **English dev** after each epoch and keeps the best epoch by entity F1.

In [ ]:
use_cuda = torch.cuda.is_available()
args = TrainingArguments(
    output_dir=str(RUN_DIR / "trainer"),
    learning_rate=cfg["learning_rate"],
    num_train_epochs=cfg["num_train_epochs"],
    per_device_train_batch_size=cfg["per_device_train_batch_size"],
    per_device_eval_batch_size=cfg["per_device_eval_batch_size"],
    warmup_ratio=cfg["warmup_ratio"],
    weight_decay=cfg["weight_decay"],
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    save_only_model=True,          # no optimizer state: ~1/3 the size on Drive
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    group_by_length=True,          # batches of similar length -> far less padding
    fp16=use_cuda,
    logging_steps=50,
    report_to="none",
    seed=SEED,
    dataloader_num_workers=2 if use_cuda else 0,
)
mem_cb = cost.PeakMemoryCallback()
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_sets["en_dev"],
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=metrics.compute_metrics,
    callbacks=[mem_cb],
)

cost.reset_peak_memory()
with cost.Timer() as t:
    train_out = trainer.train()
print(f"train time {t.seconds / 60:.1f} min | peak memory {mem_cb.peak_gb:.2f} GB")

In [ ]:
# English-dev F1 per epoch (the model-selection signal)
hist = pd.DataFrame([h for h in trainer.state.log_history if "eval_f1" in h])
hist[["epoch", "eval_loss", "eval_precision", "eval_recall", "eval_f1"]].round(4)

## 6. Evaluate

Best English-dev checkpoint, scored on English dev, English test (sanity check) and **Swahili test (zero-shot)**.

In [ ]:
# group_by_length also shuffles *eval* batches in transformers 4.5x; turn it off so predictions
# come back in dataset order and can be matched to sentences for error analysis.
trainer.args.group_by_length = False

results, sw_pred_out = {}, None
for name, ds in eval_sets.items():
    out = trainer.predict(ds)
    y_true, y_pred = metrics.align_predictions(out.predictions, out.label_ids)
    results[name] = metrics.ner_scores(y_true, y_pred)
    if name == "sw_test":
        sw_pred_out = (y_true, y_pred)

cols = ["precision", "recall", "f1"] + [f"{t}_f1" for t in data.ENTITY_TYPES]
(pd.DataFrame(results).T[cols] * 100).round(2)

## 7. Save cost + metrics

In [ ]:
ckpt_mb = cost.save_trainable(trainer.model, RUN_DIR / "trainable.safetensors")
record = {
    "run": RUN_NAME, "method": METHOD, "seed": SEED, "setting": "zero-shot", "config": cfg,
    "max_train": MAX_TRAIN, "best_epoch_en_dev": float(hist.loc[hist.eval_f1.idxmax(), "epoch"]),
    "cost": {**params, "peak_memory_gb": mem_cb.peak_gb, "train_time_min": t.seconds / 60,
             "checkpoint_mb": ckpt_mb, "device": cost.device_name(), "train_steps": train_out.global_step},
    "metrics": results,
    "versions": {"torch": torch.__version__, "transformers": transformers.__version__},
}
(RESULTS_DIR / f"{RUN_NAME}.json").write_text(json.dumps(record, indent=2))

# Swahili predictions, for error analysis later
y_true, y_pred = sw_pred_out
with open(RESULTS_DIR / f"{RUN_NAME}_sw_test_predictions.jsonl", "w", encoding="utf-8") as f:
    for r, g, p in zip(sw["test"], y_true, y_pred):
        f.write(json.dumps({"tokens": r["tokens"][:len(g)], "gold": g, "pred": p}, ensure_ascii=False) + "\n")

pd.Series(record["cost"])

In [ ]:
# A few Swahili test sentences the model got wrong (gold vs. predicted, entity tokens only)
wrong = [(r, g, p) for r, g, p in zip(sw["test"], y_true, y_pred) if g != p]
print(f"{len(wrong)}/{len(y_true)} Swahili test sentences have at least one tag error\n")
for r, g, p in wrong[:5]:
    rows = [(w, gt, pt) for w, gt, pt in zip(r["tokens"], g, p) if gt != "O" or pt != "O"]
    print(" ".join(r["tokens"][:len(g)]))
    print(pd.DataFrame(rows, columns=["token", "gold", "pred"]).to_string(index=False), "\n")